### Split data into bach of 500 for LLM extraction

In [1]:
import pandas as pd
import os
import math

# Split apple_news_data.csv by year (and batch if > 300 articles)

INPUT_FILE  = '../data/apple_news_data.csv'
OUTPUT_DIR  = '../data/splits/'

os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE  = 500
AAPL_VARIANTS = ['AAPL', 'AAPL.US', 'AAPL.BA', 'AAPL.MX', 'AAPL.SN', 'AAPL.NEO']

# Load 
print("Loading dataset...")
df = pd.read_csv(INPUT_FILE)
df['date'] = pd.to_datetime(df['date'], utc=True)
print(f"✓ Loaded {len(df):,} total articles")

# Filter AAPL
df = df[df['symbols'].str.contains('|'.join(AAPL_VARIANTS), case=False, na=False)].copy()
print(f"✓ After AAPL filter: {len(df):,} articles")

# Build text column
df['text'] = df.apply(
    lambda r: (f"{r['title']}. {r['content']}" if pd.notna(r['content']) else str(r['title']))[:2000],
    axis=1
)
df = df.dropna(subset=['text']).copy()

# Extract year
df['year'] = df['date'].dt.year
years      = sorted(df['year'].unique())
print(f"✓ Years found: {years}\n")

# Split and save
summary = []

for year in years:
    df_year = df[df['year'] == year].copy().reset_index(drop=True)
    total   = len(df_year)

    if total == 0:
        continue

    if total <= BATCH_SIZE:
        out_path = os.path.join(OUTPUT_DIR, f'aapl_{year}.csv')
        df_year.to_csv(out_path, index=False)
        print(f"  {year}: {total:>5,} articles  →  aapl_{year}.csv")
        summary.append({'year': year, 'batch': None, 'articles': total, 'file': out_path})

    else:
        total_batches = math.ceil(total / BATCH_SIZE)
        print(f"  {year}: {total:>5,} articles  →  {total_batches} batches of {BATCH_SIZE}")

        for b in range(total_batches):
            start     = b * BATCH_SIZE
            end       = min(start + BATCH_SIZE, total)
            df_batch  = df_year.iloc[start:end].copy().reset_index(drop=True)
            out_path  = os.path.join(OUTPUT_DIR, f'aapl_{year}_batch{b+1}of{total_batches}.csv')
            df_batch.to_csv(out_path, index=False)
            print(f"    Batch {b+1}/{total_batches}: articles {start+1}–{end}  →  aapl_{year}_batch{b+1}of{total_batches}.csv")
            summary.append({'year': year, 'batch': f'{b+1}of{total_batches}', 'articles': len(df_batch), 'file': out_path})

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
summary_df = pd.DataFrame(summary)
print(summary_df[['year', 'batch', 'articles', 'file']].to_string(index=False))

total_files = len(summary_df)
print(f"\n✓ Total files created : {total_files}")
print(f"✓ Output directory    : {os.path.abspath(OUTPUT_DIR)}")

summary_path = os.path.join(OUTPUT_DIR, 'split_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"✓ Summary saved to    : {summary_path}")

Loading dataset...
✓ Loaded 29,752 total articles
✓ After AAPL filter: 29,735 articles
✓ Years found: [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

  2016:     1 articles  →  aapl_2016.csv
  2017:     3 articles  →  aapl_2017.csv
  2018:     7 articles  →  aapl_2018.csv
  2019:    41 articles  →  aapl_2019.csv
  2020:   301 articles  →  aapl_2020.csv
  2021: 7,512 articles  →  16 batches of 500
    Batch 1/16: articles 1–500  →  aapl_2021_batch1of16.csv
    Batch 2/16: articles 501–1000  →  aapl_2021_batch2of16.csv
    Batch 3/16: articles 1001–1500  →  aapl_2021_batch3of16.csv
    Batch 4/16: articles 1501–2000  →  aapl_2021_batch4of16.csv
    Batch 5/16: articles 2001–2500  →  aapl_2021_batch5of16.csv
    Batch 6/16: articles 2501–3000  →  aapl_2021_batch6of16.csv
    Batch 7/16: articles 3001–3500  →  aapl_2021_batch7of16.csv
    Batch 8/16: articles 3501–4000  →  aapl_2021_batch8of16